In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 5.7 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://exterior-everyone-gander.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://exterior-everyone-gander.ngrok-free.dev


True

In [ ]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        response_modalities=["TEXT"],
    )
)

In [ ]:
def stateful_query(payload):
    response = chat.send_message(message=payload)#可以接上一個問題繼續回答
    return response.text

In [ ]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Ming Hsin University of Science and Technology），簡稱明新科大，是一所位於臺灣新竹縣新豐鄉的私立科技大學。它以其悠久的歷史、務實的辦學理念和卓越的產學合作而聞名。

以下是明新科技大學的簡要介紹：

1.  **歷史悠久**：學校創立於1966年，前身為「明新工業專科學校」，於2002年正式改制為「明新科技大學」。其深厚的辦學歷史，使其在新竹地區乃至全國技職體系中，都佔有一席之地。

2.  **地理位置優越**：明新科大鄰近新竹科學園區、湖口工業區等重要產業聚落，這使得學校在推動產學合作、提供學生實習與就業機會方面具有得天獨厚的優勢。便利的交通網絡也方便學生的通勤。

3.  **務實致用導向**：作為一所科技大學，明新科大秉持「明德、新民、致知、力行」的校訓，強調理論與實務並重，課程設計緊密結合產業需求，致力於培養具備專業技能和職場競爭力的優質人才。

4.  **學術領域多元**：學校設有多個學院，涵蓋了工學、管理學、服務產業、設計等多元領域，例如工程學院、管理學院、服務事業學院、人文社會與設計學院。提供學士班、碩士班及進修部等不同學制，滿足學生多元的學習需求。

5.  **產學合作成果豐碩**：明新科大積極與企業界建立緊密合作關係，推動產業專班、企業實習、共同研發等計畫。學生在學期間就能接觸產業實務，累積實作經驗，畢業後能快速與職場接軌，因此畢業生就業率表現良好，深受企業歡迎。

6.  **校園環境與設施**：學校擁有現代化的教學大樓、實驗室、圖書館、體育館、學生宿舍等完善設施，提供學生舒適便捷的學習與生活環境。

總體而言，明新科技大學是一所強調實用技能、深耕在地產業、並致力於培育具備未來產業所需人才的科技學府，是北台灣技職教育的重要基地之一。


In [ ]:
result2 = stateful_query("校長是誰？")
print(result2)

明新科技大學現任校長是 **劉國偉** 博士 (Dr. Liu Kuo-Wei)。


In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit


BODY:  {"destination":"U4bf2ca05145c0285eed4eb00c147e436","events":[{"type":"message","message":{"type":"text","id":"615032012776145124","quoteToken":"ozoeWUAB16HIj_pBnyRggN1cQkO7WD6Gav00Q2cUlDApVj1zhclE0hSb0qEC3JiIAgljJwe-PGzFiLvRQqXavxJuesgCK0Ez8myAWRCuiG3aMVx97rjaHY2XHhVnDO_zgvrqoTEUioSEwAJdawGCUg","markAsReadToken":"vJq1pimlokBNNGEZ8dSZ-6sDwC8rFiTYmDOObuJaTZjPmO_GkVVoA2ICyfam-tOYxbG8F7TrJv68MWkBgS_-1C-5YAo4apbBIpm9TYsVc1d02iOevh-jtacL0fkAXJ63ynPj62ahBlhI7fMAZhml_SggV2WvPct3mi-POAvUGNiieWRJsaoD5tMsOxfu8VgcQQy7YiHXp_qvp05s7qe1KA","text":"AI 簡介明新科大，30字以內"},"webhookEventId":"01KS6T4V449G68XWRJBPZ5PATJ","deliveryContext":{"isRedelivery":false},"timestamp":1779418950641,"source":{"type":"user","userId":"U7de6d64df43af01c22a89a5ae823b57a"},"replyToken":"709c35e117f4430b8c4222234b8ae75e","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 03:02:34] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U4bf2ca05145c0285eed4eb00c147e436","events":[{"type":"message","message":{"type":"text","id":"615032051615138156","quoteToken":"EJkKFk3r33j02qRbPIwuvKNITKqZcAoZgaXKpkVimulk-yZXMQnSLmYxtg5PdkFqxOUNL5-X_gmWgcR1szN5eo48-rSrC3ARqKH_R8NA9d9SCd2zZJ4tVNoWVR-KsIcxXgqu_dIdeywp_3WlH0lnTg","markAsReadToken":"D9s9b6Aky39kwsDVGJx_u2jY26MxP5piBS-VxZLbMAtpXg195X4I-TqoHC3joD4pmzqVvKnmt8cBm7yN6A5t5uMnmv-U5RxDXPTFT-KCXfbiJEkXpgHyJ0w6xXEa9KVY5eF0mOboIp3-_Ib1S9teHZIP1-tByLXU5aUGHWia9j6lvx2PnikrVQ3ZR8D5D2mATQCOuv3bCkG1Qbqx6UUxSQ","text":"AI 校長是誰"},"webhookEventId":"01KS6T5HPX6R9NNWAY29R77AXD","deliveryContext":{"isRedelivery":false},"timestamp":1779418973656,"source":{"type":"user","userId":"U7de6d64df43af01c22a89a5ae823b57a"},"replyToken":"fa2fb74b935d40538c4fb328fba3816e","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 03:02:55] "POST / HTTP/1.1" 200 -
